<a href="https://colab.research.google.com/github/Rei-stark/semaglutide/blob/main/Semaglutida.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from getpass import getpass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from supabase import create_client

# Informe as credenciais apenas durante a execução. Nunca as salve no notebook.
SUPABASE_URL = os.getenv("SUPABASE_URL") or input("URL do Supabase: ").strip()
SUPABASE_KEY = os.getenv("SUPABASE_KEY") or getpass("Chave do Supabase: ")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
USUARIOS = {
    "Glauciana": "glaucianaffonseca@gmail.com",
    "Reinaldo": "reinaldogalvao@gmail.com",
}


def carregar_historico(email):
    perfil = (
        supabase.table("utilizadores")
        .select("id, nome, peso_inicial")
        .eq("email", email)
        .execute()
    )
    if not perfil.data:
        raise ValueError(f"Perfil não encontrado: {email}")

    user_id = perfil.data[0]["id"]
    historico = (
        supabase.table("registos_diarios")
        .select("data_registo, peso")
        .eq("user_id", user_id)
        .order("data_registo")
        .execute()
    )
    dados = pd.DataFrame(historico.data)
    if len(dados) < 4:
        raise ValueError(f"Histórico insuficiente para {email}: são necessários 4 registros.")

    dados["data_registo"] = pd.to_datetime(dados["data_registo"])
    dados = dados.sort_values("data_registo").dropna(subset=["peso"]).copy()
    dados["dias_tratamento"] = (
        dados["data_registo"] - dados["data_registo"].min()
    ).dt.days
    return dados, perfil.data[0]


def projetar(dados):
    X = dados[["dias_tratamento"]]
    y = dados["peso"].astype(float)

    modelo_ridge = make_pipeline(PolynomialFeatures(degree=2), Ridge(alpha=10.0))
    modelo_svr = make_pipeline(
        StandardScaler(),
        SVR(kernel="rbf", C=10.0, gamma="scale", epsilon=0.1),
    )
    modelo_ridge.fit(X, y)
    modelo_svr.fit(X, y)

    previsao_ridge_historico = modelo_ridge.predict(X)
    previsao_svr_historico = modelo_svr.predict(X)
    dias_futuros = np.arange(dados["dias_tratamento"].max() + 1, dados["dias_tratamento"].max() + 31)
    previsao_ridge_futura = modelo_ridge.predict(pd.DataFrame({"dias_tratamento": dias_futuros}))
    previsao_svr_futura = modelo_svr.predict(pd.DataFrame({"dias_tratamento": dias_futuros}))

    return {
        "modelo_ridge": modelo_ridge,
        "modelo_svr": modelo_svr,
        "erro_ridge": mean_absolute_error(y, previsao_ridge_historico),
        "erro_svr": mean_absolute_error(y, previsao_svr_historico),
        "dias_futuros": dias_futuros,
        "previsao_ridge": previsao_ridge_futura,
        "previsao_svr": previsao_svr_futura,
    }


resultados = {}
for nome, email in USUARIOS.items():
    historico, perfil = carregar_historico(email)
    resultados[nome] = {
        "historico": historico,
        "perfil": perfil,
        "projecao": projetar(historico),
    }

fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=False)
axes = np.atleast_1d(axes)

for ax, (nome, resultado) in zip(axes, resultados.items()):
    dados = resultado["historico"]
    projecao = resultado["projecao"]
    ultimo_dia = dados["dias_tratamento"].max()
    datas_futuras = dados["data_registo"].max() + pd.to_timedelta(
        projecao["dias_futuros"] - ultimo_dia, unit="D"
    )

    ax.scatter(
        dados["dias_tratamento"],
        dados["peso"],
        color="black",
        s=35,
        label="Peso real",
        zorder=5,
    )
    ax.plot(
        dados["dias_tratamento"],
        projecao["modelo_ridge"].predict(dados[["dias_tratamento"]]),
        color="blue",
        linestyle="--",
        label="Previsão Ridge (+30 dias)",
    )
    ax.plot(
        dados["dias_tratamento"],
        projecao["modelo_svr"].predict(dados[["dias_tratamento"]]),
        color="green",
        linestyle="--",
        label="Previsão SVR (+30 dias)",
    )
    ax.plot(
        projecao["dias_futuros"],
        projecao["previsao_ridge"],
        color="blue",
        linestyle="--",
    )
    ax.plot(
        projecao["dias_futuros"],
        projecao["previsao_svr"],
        color="green",
        linestyle="--",
    )
    ax.set_title(f"Modelos avançados de ML - {nome}")
    ax.set_xlabel("Dias de tratamento")
    ax.set_ylabel("Peso (kg)")
    ax.grid(True, alpha=0.25)
    ax.legend()

    print(f"\n{nome}")
    print(f"Erro médio - Ridge Polynomial: {projecao['erro_ridge']:.3f} kg")
    print(f"Erro médio - SVR (RBF): {projecao['erro_svr']:.3f} kg")
    print(f"Ridge em 30 dias: {projecao['previsao_ridge'][-1]:.1f} kg")
    print(f"SVR em 30 dias: {projecao['previsao_svr'][-1]:.1f} kg")

plt.tight_layout()
plt.show()